In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
import joblib

# Scikit-learn libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# TensorFlow/Keras libraries (for Neural Network benchmark)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# --- CONFIGURATION & REPRODUCIBILITY ---
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)

# Define file paths based on the prescribed folder structure
DATA_PATH = '../data/Crop_Production_Statistics.csv'  # Updated path to data folder
MODEL_DIR = '../model'  # Updated path to model folder
MODEL_FILENAME = os.path.join(MODEL_DIR, 'final_xgb_model.pkl')
SCALER_FILENAME = os.path.join(MODEL_DIR, 'scaler.pkl')

# Constants
TARGET_COLUMN = 'Yield' 
BASE_TEMP = 10 # Base temperature for GDD calculation in °C

# Ensure model directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

print("Setup complete. Libraries imported and paths configured.")
print(f"Data path: {DATA_PATH}")
print(f"Model directory: {MODEL_DIR}")

Setup complete. Libraries imported and paths configured.
Data path: ../data/Crop_Production_Statistics.csv
Model directory: ../model


In [3]:
print("--- 1. DATA EXPLORATION & PREPROCESSING ---")

try:
    df = pd.read_csv(DATA_PATH)  # Using the updated DATA_PATH variable
    print(f"✅ Successfully loaded data from: {DATA_PATH}")
    print(f"Dataset shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: {DATA_PATH} not found. Please ensure the file exists in the data directory.")
    raise

# Initial Cleaning and Target Calculation
df.columns = df.columns.str.strip().str.replace(' ', '_')
df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

# Rename columns and calculate target yield (Quintals/Hectare)
df.rename(columns={'Crop_Year': 'Year', 'Area': 'Area_Hectares', 'Production': 'Production_Tonnes'}, inplace=True)
df['Yield_Quintals'] = df[TARGET_COLUMN] * 10 
print(f"Target variable calculated. Initial shape: {df.shape}")

# --- Mock essential features missing in the raw CSV (Crucial for ML) ---
# NOTE: These features would come from your Weather/Soil APIs and be merged in a real system.
n_samples = len(df)
np.random.seed(42)  # For reproducible results
df['temperature_avg'] = np.random.uniform(20, 35, n_samples)
df['rainfall_mm_cumulative'] = np.random.uniform(100, 1000, n_samples)
df['soil_ph'] = np.random.uniform(5.5, 7.5, n_samples)
df['soil_nitrogen'] = np.random.uniform(50, 150, n_samples)
df['soil_phosphorus'] = np.random.uniform(20, 80, n_samples)
df['soil_potassium'] = np.random.uniform(10, 50, n_samples)
df['planting_date'] = pd.to_datetime(df['Year'].astype(str) + '-05-01', errors='coerce')
print("Mock weather and soil features added.")
print(f"Sample of added features:\n{df[['temperature_avg', 'rainfall_mm_cumulative', 'soil_ph']].head()}")

--- 1. DATA EXPLORATION & PREPROCESSING ---
✅ Successfully loaded data from: ../data/Crop_Production_Statistics.csv
Dataset shape: (345336, 8)
Target variable calculated. Initial shape: (345336, 9)
✅ Successfully loaded data from: ../data/Crop_Production_Statistics.csv
Dataset shape: (345336, 8)
Target variable calculated. Initial shape: (345336, 9)
Mock weather and soil features added.
Sample of added features:
   temperature_avg  rainfall_mm_cumulative   soil_ph
0        25.618102              790.310269  5.811041
1        34.260715              862.165020  6.696400
2        30.979909              786.127351  5.769896
3        28.979877              479.100156  6.848426
4        22.340280              961.805565  6.481542
Mock weather and soil features added.
Sample of added features:
   temperature_avg  rainfall_mm_cumulative   soil_ph
0        25.618102              790.310269  5.811041
1        34.260715              862.165020  6.696400
2        30.979909              786.127351 

In [4]:
Q1 = df['Area_Hectares'].quantile(0.25)
Q3 = df['Area_Hectares'].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df['Area_Hectares'] < (Q1 - 3 * IQR)) | (df['Area_Hectares'] > (Q3 + 3 * IQR)))]
print("Outliers handled on Area_Hectares.")

# 1.3 Categorical Encoding
le = LabelEncoder()
df['State_Encoded'] = le.fit_transform(df['State'])
df['District_Encoded'] = le.fit_transform(df['District'])

# One-Hot Encode Season and Crop
df = pd.get_dummies(df, columns=['Season', 'Crop'], drop_first=True, prefix=['Season', 'Crop'])
print("Categorical features encoded.")

# --- 2. FEATURE ENGINEERING ---
# 2.1 Growing Degree Days (GDD)
df['GDD'] = (df['temperature_avg'] - BASE_TEMP).clip(lower=0) 

# 2.2 Nutrient Balance Ratios (N:P:K)
df['NPK_Ratio_N'] = df['soil_nitrogen'] / (df['soil_phosphorus'] + 1e-6)
df['NPK_Ratio_K'] = df['soil_potassium'] / (df['soil_phosphorus'] + 1e-6)

# 2.3 Interaction Features
df['Rain_x_Nitrogen'] = df['rainfall_mm_cumulative'] * df['soil_nitrogen']


# 2.4 Feature Selection (Dropping non-numeric/redundant columns)
features_to_drop = [
    'State', 'District', 'Year', 'Yield', 
    'Production_Tonnes', TARGET_COLUMN, 'planting_date', 
    'temperature_avg' 
]

X = df.drop(columns=[col for col in features_to_drop if col in df.columns])
Y = df['Yield_Quintals']

print(f"Final features count: {len(X.columns)}")

# 2.5 Normalization and Split
numerical_cols = X.select_dtypes(include=np.number).columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
print("Numerical features scaled using StandardScaler.")

# Split data: 70/15/15 (Train/Validation/Test)
X_temp, X_test, y_temp, y_test = train_test_split(X, Y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.15 / 0.85), random_state=42)

Outliers handled on Area_Hectares.
Categorical features encoded.
Final features count: 72
Categorical features encoded.
Final features count: 72
Numerical features scaled using StandardScaler.
Numerical features scaled using StandardScaler.


In [ ]:
# --- ADVANCED FEATURE ENGINEERING FOR AGRICULTURE ---
print("--- 2. ADVANCED FEATURE ENGINEERING ---")

# 2.1 Advanced Agricultural Features
# Temperature-based features
df['temp_stress_high'] = np.where(df['temperature_avg'] > 32, df['temperature_avg'] - 32, 0)
df['temp_stress_low'] = np.where(df['temperature_avg'] < 15, 15 - df['temperature_avg'], 0)
df['optimal_temp_range'] = np.where((df['temperature_avg'] >= 20) & (df['temperature_avg'] <= 30), 1, 0)

# Rainfall patterns (more realistic for agriculture)
df['rainfall_adequacy'] = np.where((df['rainfall_mm_cumulative'] >= 400) & (df['rainfall_mm_cumulative'] <= 800), 1, 0)
df['drought_stress'] = np.where(df['rainfall_mm_cumulative'] < 300, 300 - df['rainfall_mm_cumulative'], 0)
df['flood_risk'] = np.where(df['rainfall_mm_cumulative'] > 1000, df['rainfall_mm_cumulative'] - 1000, 0)

# Advanced soil features
df['soil_health_score'] = (
    (df['soil_ph'] >= 6.0) & (df['soil_ph'] <= 7.5)
).astype(int) * 0.3 + (df['soil_nitrogen'] / 150) * 0.4 + (df['soil_phosphorus'] / 80) * 0.3

# NPK balance (critical for agriculture)
df['NPK_total'] = df['soil_nitrogen'] + df['soil_phosphorus'] + df['soil_potassium']
df['N_P_ratio'] = df['soil_nitrogen'] / (df['soil_phosphorus'] + 1e-6)
df['N_K_ratio'] = df['soil_nitrogen'] / (df['soil_potassium'] + 1e-6)
df['P_K_ratio'] = df['soil_phosphorus'] / (df['soil_potassium'] + 1e-6)

# Climate-soil interactions
df['climate_soil_index'] = (df['temperature_avg'] * df['rainfall_mm_cumulative'] * df['soil_health_score']) / 10000

# Seasonal adjustments based on crop year patterns
df['year_trend'] = df['Year'] - df['Year'].min()
df['cyclical_pattern'] = np.sin(2 * np.pi * df['Year'] / 7)  # 7-year agricultural cycle

print("✅ Advanced agricultural features created")
print(f"New feature count: {len([col for col in df.columns if col not in ['State', 'District', 'Year', 'Yield', 'Production_Tonnes', TARGET_COLUMN, 'planting_date']])}")

In [ ]:
# --- IMPROVED MODEL TRAINING WITH ENSEMBLE ---
print("--- 3. ENSEMBLE MODEL TRAINING ---")

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import VotingRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import lightgbm as lgb

def comprehensive_evaluate_model(model, X_set, y_set, name):
    """Enhanced evaluation with more agricultural metrics"""
    if isinstance(model, Sequential):
        y_pred = model.predict(X_set, verbose=0).flatten()
    else:
        y_pred = model.predict(X_set)
        
    r2 = r2_score(y_set, y_pred)
    rmse = np.sqrt(mean_squared_error(y_set, y_pred))
    mae = mean_absolute_error(y_set, y_pred)
    mape = mean_absolute_percentage_error(y_set, y_pred) * 100
    
    # Agricultural specific metrics
    yield_accuracy_10pct = np.mean(np.abs(y_pred - y_set) / y_set <= 0.10) * 100  # Within 10% accuracy
    yield_accuracy_20pct = np.mean(np.abs(y_pred - y_set) / y_set <= 0.20) * 100  # Within 20% accuracy
    
    return {
        'Model': name, 
        'R2': r2, 
        'RMSE': rmse, 
        'MAE': mae,
        'MAPE': mape,
        'Accuracy_10%': yield_accuracy_10pct,
        'Accuracy_20%': yield_accuracy_20pct
    }

# Improved XGBoost with hyperparameter tuning
print("Training optimized XGBoost...")
xgb_optimized = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# LightGBM (often better for tabular data)
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Gradient Boosting
print("Training Gradient Boosting...")
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=42
)

# Train individual models
models_dict = {
    'XGBoost_Optimized': xgb_optimized,
    'LightGBM': lgb_model,
    'GradientBoosting': gb_model
}

trained_models = {}
model_results = []

for name, model in models_dict.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Evaluate on validation set
    val_results = comprehensive_evaluate_model(model, X_val, y_val, f"{name}_Val")
    model_results.append(val_results)
    print(f"{name} - Val R2: {val_results['R2']:.4f}, RMSE: {val_results['RMSE']:.2f}")

print("✅ Individual models trained")

In [ ]:
# --- ENSEMBLE MODEL CREATION ---
print("--- 4. CREATING ENSEMBLE MODEL ---")

# Create ensemble using voting regressor (weighted by validation performance)
ensemble_model = VotingRegressor([
    ('xgb', trained_models['XGBoost_Optimized']),
    ('lgb', trained_models['LightGBM']),
    ('gb', trained_models['GradientBoosting'])
], weights=[0.4, 0.35, 0.25])  # Weights based on typical performance

print("Training ensemble model...")
ensemble_model.fit(X_train, y_train)

# Evaluate ensemble
ensemble_val_results = comprehensive_evaluate_model(ensemble_model, X_val, y_val, "Ensemble_Val")
model_results.append(ensemble_val_results)

print(f"Ensemble Model - Val R2: {ensemble_val_results['R2']:.4f}, RMSE: {ensemble_val_results['RMSE']:.2f}")

# Select best model based on comprehensive metrics
best_score = -1
best_model_name = ""
best_model = None

print("\n--- MODEL COMPARISON ---")
results_df = pd.DataFrame(model_results)
print(results_df.round(4))

# Select best model (prioritizing R2 and agricultural accuracy)
for result in model_results:
    # Composite score: R2 * 0.4 + (1-MAPE/100) * 0.3 + Accuracy_20% * 0.3
    composite_score = (result['R2'] * 0.4 + 
                      (1 - result['MAPE']/100) * 0.3 + 
                      result['Accuracy_20%']/100 * 0.3)
    
    if composite_score > best_score:
        best_score = composite_score
        best_model_name = result['Model']
        if 'Ensemble' in result['Model']:
            best_model = ensemble_model
        elif 'XGBoost' in result['Model']:
            best_model = trained_models['XGBoost_Optimized']
        elif 'LightGBM' in result['Model']:
            best_model = trained_models['LightGBM']
        elif 'GradientBoosting' in result['Model']:
            best_model = trained_models['GradientBoosting']

print(f"\n🏆 Best Model Selected: {best_model_name}")
print(f"Composite Score: {best_score:.4f}")

# Cross-validation for robustness check
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')
print(f"Cross-validation R2: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

print("✅ Best model selected based on comprehensive evaluation")

In [5]:
def evaluate_model(model, X_set, y_set, name):
    if isinstance(model, Sequential):
        y_pred = model.predict(X_set, verbose=0).flatten()
    else:
        y_pred = model.predict(X_set)
        
    r2 = r2_score(y_set, y_pred)
    rmse = np.sqrt(mean_squared_error(y_set, y_pred))
    mape = np.mean(np.abs((y_set - y_pred) / y_set.replace(0, np.nan).dropna())) * 100
    return {'Model': name, 'R2': r2, 'RMSE': rmse, 'MAPE': mape}

# Train XGBoost Model (Best Performer for Production)
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror', n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42
).fit(X_train, y_train)

# Select best model
best_model = xgb_model 

In [6]:
# --- FINAL MODEL EVALUATION AND SAVING ---
print("--- 5. FINAL MODEL EVALUATION ---")

# Test set evaluation with comprehensive metrics
test_results = comprehensive_evaluate_model(best_model, X_test, y_test, f"{best_model_name}_Test")

print(f"\n🎯 FINAL MODEL PERFORMANCE:")
print(f"Model Type: {best_model_name}")
print(f"Test R²: {test_results['R2']:.4f}")
print(f"Test RMSE: {test_results['RMSE']:.2f} quintals")
print(f"Test MAE: {test_results['MAE']:.2f} quintals")
print(f"Test MAPE: {test_results['MAPE']:.2f}%")
print(f"Predictions within 10% accuracy: {test_results['Accuracy_10%']:.1f}%")
print(f"Predictions within 20% accuracy: {test_results['Accuracy_20%']:.1f}%")

# Feature importance analysis
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n📊 TOP 10 MOST IMPORTANT FEATURES:")
    print(feature_importance.head(10).to_string(index=False))
elif hasattr(best_model, 'estimators_'):  # For ensemble
    # Average feature importance across ensemble
    importances = []
    for estimator in best_model.estimators_:
        if hasattr(estimator, 'feature_importances_'):
            importances.append(estimator.feature_importances_)
    
    if importances:
        avg_importance = np.mean(importances, axis=0)
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': avg_importance
        }).sort_values('importance', ascending=False)
        
        print(f"\n📊 TOP 10 MOST IMPORTANT FEATURES (Ensemble Average):")
        print(feature_importance.head(10).to_string(index=False))

# Save the improved model
improved_model_filename = os.path.join(MODEL_DIR, 'improved_ensemble_model.pkl')
improved_scaler_filename = os.path.join(MODEL_DIR, 'improved_scaler.pkl')

joblib.dump(best_model, improved_model_filename)
joblib.dump(scaler, improved_scaler_filename)

print(f"\n✅ Improved model saved to: {improved_model_filename}")
print(f"✅ Improved scaler saved to: {improved_scaler_filename}")

# Performance comparison
improvement_r2 = test_results['R2'] - 0.9364  # Previous R2 score
improvement_rmse = 2160.12 - test_results['RMSE']  # Previous RMSE

print(f"\n📈 IMPROVEMENT SUMMARY:")
print(f"R² improvement: {improvement_r2:+.4f}")
print(f"RMSE improvement: {improvement_rmse:+.2f} quintals")
print(f"Agricultural accuracy (±20%): {test_results['Accuracy_20%']:.1f}%")


Selected Production Model: XGBRegressor
Test Set R2: 0.9364 | Test Set RMSE: 2160.12

✅ Model saved to: ../model\final_xgb_model.pkl
✅ Scaler saved to: ../model\scaler.pkl


In [7]:
print("\n--- 5. RECOMMENDATION ENGINE LOGIC ---")

# Mock Current Sensor Readings (INPUTS TO RECOMMENDATION ENGINE) 
# In production, these values come from the Node.js API call (from MongoDB/Weather Service)
mock_current_rainfall = 15 # low rain (mm)
mock_soil_N = 55 # low nitrogen (units)
mock_soil_P = 30 # low phosphorus (units)
mock_soil_moisture = 50 # % of field capacity
mock_current_humidity = 92 # high humidity
mock_crop_stage = "Flowering" # Critical stage
predicted_yield = best_model.predict(X_test.iloc[0].values.reshape(1, -1))[0]
historical_avg_yield = df['Yield_Quintals'].mean()


def generate_recommendations(prediction, historical_avg, rainfall, soil_moisture, soil_N, soil_P, humidity, crop_stage):
    """Generates a list of actionable recommendations based on ML prediction and rules."""
    recommendations = []
    
    # Define optimal values for the current crop type (Example for Wheat/Rice)
    OPTIMAL_N = 120
    OPTIMAL_P = 60
    
    # --- A. IRRIGATION RECOMMENDATIONS (Water Stress) ---
    # Rule 1: Immediate Irrigation Need (Low moisture, High risk)
    water_deficit = 85 - soil_moisture # Assume 85% is ideal
    if water_deficit > 30: # If moisture is below 55%
        advice = f"CRITICAL: Initiate heavy irrigation immediately. Soil moisture is at {soil_moisture}%, indicating severe water stress."
        recommendations.append({'type': 'Irrigation', 'advice': advice, 'metric': f'Water Deficit: {water_deficit:.0f}%', 'priority': 'Urgent'})
    # Rule 2: Preventative Irrigation (Low yield prediction)
    elif prediction < historical_avg * 0.90 and rainfall < 20: 
        advice = f"HIGH PRIORITY: Increase next irrigation volume by 15%. Predicted yield may drop due to low recent rainfall."
        recommendations.append({'type': 'Irrigation', 'advice': advice, 'metric': '+15% Volume', 'priority': 'High'})
    
    # --- B. FERTILIZATION RECOMMENDATIONS (Nutrient Deficit) ---
    # Rule 3: Nitrogen Deficit
    N_deficit = OPTIMAL_N - soil_N
    if N_deficit > 40: # If N is 40 units below ideal
        dosage = N_deficit * 2.2 # Example conversion factor to kg/ha Urea
        advice = f"ACTION: Soil Nitrogen is low. Apply {dosage:.1f} kg/ha of Urea within 7 days to support {crop_stage} growth."
        recommendations.append({'type': 'Fertilization', 'advice': advice, 'metric': f'{dosage:.1f}kg Urea/Ha', 'priority': 'High'})

    # Rule 4: NPK Imbalance (e.g., too much N relative to P)
    if soil_N > OPTIMAL_N * 1.5 and soil_P < OPTIMAL_P * 0.5:
        advice = f"WARNING: NPK imbalance detected. Excess Nitrogen will impact flower setting. Apply Phosphate supplement immediately."
        recommendations.append({'type': 'Fertilization', 'advice': advice, 'metric': 'NPK Imbalance', 'priority': 'Medium'})

    # --- C. PEST CONTROL RECOMMENDATIONS (Weather Trigger) ---
    # Rule 5: Fungal/Pest Risk based on environment
    if humidity > 90 and crop_stage in ["Flowering", "Fruiting"]: 
         advice = "RISK ALERT: High humidity and mild temperatures favor Fungal Blight. Initiate preventative scouting in the field and prepare fungicide."
         recommendations.append({'type': 'Pest Control', 'advice': advice, 'metric': 'Preemptive Action', 'priority': 'High'})
    elif humidity > 80 and crop_stage == "Vegetative":
         advice = "MONITOR: Moderate risk of pests. Keep monitoring new growth, especially the undersides of leaves."
         recommendations.append({'type': 'Pest Control', 'advice': advice, 'metric': 'Daily Scouting', 'priority': 'Low'})
         
    if not recommendations:
         recommendations.append({'type': 'Status', 'advice': 'All parameters are within optimal ranges. Continue current farming practice.', 'metric': 'Optimal', 'priority': 'Low'})


    return recommendations

recommendations = generate_recommendations(
    predicted_yield, 
    historical_avg_yield, 
    mock_current_rainfall, 
    mock_soil_moisture, 
    mock_soil_N, 
    mock_soil_P, 
    mock_current_humidity, 
    mock_crop_stage
)

print("--- Sample Recommendations Generated (JSON Output) ---")
print(json.dumps(recommendations, indent=4))


--- 5. RECOMMENDATION ENGINE LOGIC ---
--- Sample Recommendations Generated (JSON Output) ---
[
    {
        "type": "Irrigation",
        "advice": "CRITICAL: Initiate heavy irrigation immediately. Soil moisture is at 50%, indicating severe water stress.",
        "metric": "Water Deficit: 35%",
        "priority": "Urgent"
    },
    {
        "type": "Fertilization",
        "advice": "ACTION: Soil Nitrogen is low. Apply 143.0 kg/ha of Urea within 7 days to support Flowering growth.",
        "metric": "143.0kg Urea/Ha",
        "priority": "High"
    },
    {
        "type": "Pest Control",
        "advice": "RISK ALERT: High humidity and mild temperatures favor Fungal Blight. Initiate preventative scouting in the field and prepare fungicide.",
        "metric": "Preemptive Action",
        "priority": "High"
    }
]
